In [ ]:
# ==================== NUTRIPRO ANALYSIS - OPTIMIZED FOR VS CODE ====================
# Senior Data Analyst + Biotech Perspective

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Cài đặt hiển thị đẹp
pd.set_option('display.max_columns', None)
sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Thư viện đã import thành công!")

In [ ]:
# ====================== PHẦN 1: DATA LOADING & CLEANING ======================

file_path = 'NutriPro_PowerBI_Package.xlsx'

# Đọc 3 sheet
dim_users = pd.read_excel(file_path, sheet_name='dim_users')
fact_sales = pd.read_excel(file_path, sheet_name='fact_sales')
fact_engagement = pd.read_excel(file_path, sheet_name='fact_engagement')

print(f"dim_users: {dim_users.shape}")
print(f"fact_sales: {fact_sales.shape}")
print(f"fact_engagement: {fact_engagement.shape}")

# Data Cleaning
dim_users['diet_type'] = dim_users['diet_type'].fillna('Balanced')
dim_users['status'] = dim_users['status'].fillna('inactive')

# Chuyển đổi datetime
dim_users['signup_date'] = pd.to_datetime(dim_users['signup_date'])
dim_users['last_active_date'] = pd.to_datetime(dim_users['last_active_date'])
fact_sales['SALE DATE'] = pd.to_datetime(fact_sales['SALE DATE'])

print("✅ Data Cleaning hoàn tất!")

In [ ]:
# ====================== PHẦN 2: EDA - BIOTECH PERSPECTIVE ======================

# Phân bố chế độ ăn
plt.figure(figsize=(10,5))
sns.countplot(data=dim_users, x='diet_type', order=dim_users['diet_type'].value_counts().index)
plt.title('Phân bố Chế độ Ăn của Người dùng NutriPro')
plt.show()

# Meal logging theo diet type
if 'meal_log_count' in dim_users.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(data=dim_users, x='diet_type', y='meal_log_count')
    plt.title('Mức độ ghi nhật ký bữa ăn theo Chế độ Ăn')
    plt.ylabel('Số lần ghi Meal Log')
    plt.show()

print(dim_users.groupby('diet_type')['meal_log_count'].agg(['mean', 'median', 'count']))

In [ ]:
### Biotech Insight
- **Keto users**: Thường có churn cao hơn do giai đoạn **Keto Flu** (7-21 ngày) - cơ thể thích nghi chuyển hóa từ glucose sang ketone.
- **Balanced diet**: Dễ duy trì hơn vì phù hợp với sinh lý đại đa số.
- Ngưỡng **21 ngày** được chọn vì đây là thời gian trung bình hình thành thói quen (theo nghiên cứu Phillippa Lally).

In [ ]:
# ====================== PHẦN 3: RFM SCORING ======================

snapshot_date = pd.to_datetime('2024-12-31')

rfm = fact_sales.groupby('USER ID').agg({
    'SALE DATE': lambda x: (snapshot_date - x.max()).days,      # Recency
    'SALE ID': 'count',                                         # Frequency
    'PURCHASE AMOUNT': 'sum'                                    # Monetary
}).rename(columns={
    'SALE DATE': 'Recency',
    'SALE ID': 'Frequency',
    'PURCHASE AMOUNT': 'Monetary'
})

# RFM Score 1-5
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5])

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print(rfm.head())
print("\nTop 10 Champion Users (RFM cao):")
print(rfm.nlargest(10, 'Monetary'))

In [ ]:
# ====================== PHẦN 4: CHURN PREDICTION ======================

# Merge dữ liệu
user_analysis = dim_users.merge(rfm, left_on='USER ID', right_index=True, how='left')

# Tính churn score
user_analysis['Days_Inactive'] = (snapshot_date - user_analysis['last_active_date']).dt.days

user_analysis['Churn_Score'] = (
    (user_analysis['Days_Inactive'] > 21).astype(int) * 40 +           # 21 ngày = ngưỡng thói quen
    (user_analysis['diet_type'] == 'Keto').astype(int) * 20 +         # Keto khó duy trì
    (user_analysis.get('meal_log_count', 0) < 10).astype(int) * 25
)

user_analysis['Churn_Risk'] = pd.cut(user_analysis['Churn_Score'], 
                                     bins=[-1, 30, 60, 100], 
                                     labels=['Low', 'Medium', 'High'])

print(user_analysis['Churn_Risk'].value_counts())

In [ ]:
# ====================== EXPORT DATA ======================

import os
os.makedirs('data', exist_ok=True)

dim_users.to_csv('data/dim_users.csv', index=False)
fact_sales.to_csv('data/fact_sales.csv', index=False)
user_analysis.to_csv('data/fact_engagement_scored.csv', index=False)

print("✅ Đã xuất 3 file CSV vào thư mục 'data/' thành công!")